# HuggingFace Pipeline 入門（2026 版）

## 學習目標

1. 了解 `pipeline` 的概念：用一行程式碼完成推論的高層抽象
2. 掌握 2026 統一慣例：`device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True`
3. 理解 pipeline 執行的底層三步驟：tokenize → model forward → post-process
4. 認識 `AutoProcessor` 的角色，為多模態任務（第 05 模組）打基礎

## 前置知識

- Python 基礎語法
- Tensor 基本概念（知道 shape 即可）

## 課程銜接

- 本 notebook 是 `01-Component` 的起點
- 下一步：`../02tokenizer/01.tokenizer.ipynb`（深入 tokenizer 原理）
- 延伸：`../../05-Multimodal/`（AutoProcessor 的多模態延伸）

## 0. 版本鎖定與環境確認

2026 全 repo 統一依賴版本。固定版本可避免 API 破壞性變更造成的不可重現問題。

In [ ]:
# Install pinned dependencies for reproducibility
# Run once per environment; skip if already installed
%pip install -q \
    "transformers>=4.46" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "safetensors>=0.4" \
    "torch>=2.4" \
    sentencepiece \
    Pillow \
    requests

In [ ]:
import torch
import transformers
import accelerate

print(f"torch       : {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"accelerate  : {accelerate.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Pipeline 支援的任務總覽

`pipeline` 是 HuggingFace transformers 提供的高層 API，讓你用一行程式碼完成從文字分類到物體偵測的各種推論任務。其底層封裝了三個步驟：

1. **Tokenize / Pre-process**：將原始輸入（文字、圖像、音訊）轉換成模型可讀的 tensor
2. **Model forward**：呼叫模型取得 logits 或 hidden states
3. **Post-process**：將原始數字輸出轉換成人類可讀的結果（標籤、機率、bounding box 等）

In [ ]:
from transformers.pipelines import SUPPORTED_TASKS

print(f"pipeline 支援的任務（共 {len(SUPPORTED_TASKS)} 種）:\n")
for task_name in SUPPORTED_TASKS:
    print(f"  {task_name}")

### 常見任務中英對照

| 英文名稱 | 繁體中文名稱 |
|---|---|
| audio-classification | 音訊分類 |
| automatic-speech-recognition | 自動語音辨識 |
| text-to-audio | 文字轉音訊 |
| feature-extraction | 特徵提取 |
| text-classification | 文字分類 |
| token-classification | 標記分類（NER 等）|
| question-answering | 問答（抽取式）|
| table-question-answering | 表格問答 |
| visual-question-answering | 視覺問答 |
| document-question-answering | 文件問答 |
| fill-mask | 填充遮蔽（MLM）|
| summarization | 摘要 |
| translation | 翻譯 |
| text2text-generation | 文字到文字生成 |
| text-generation | 文字生成（自迴歸）|
| zero-shot-classification | 零樣本分類 |
| zero-shot-image-classification | 零樣本圖像分類 |
| zero-shot-audio-classification | 零樣本音訊分類 |
| image-classification | 圖像分類 |
| image-segmentation | 圖像分割 |
| image-to-text | 圖像描述 |
| object-detection | 物體偵測 |
| zero-shot-object-detection | 零樣本物體偵測 |
| depth-estimation | 深度估計 |
| video-classification | 影片分類 |
| mask-generation | 遮罩生成（SAM）|
| image-to-image | 圖像到圖像 |

## 2. 環境資源確認

In [ ]:
import subprocess
import psutil

# GPU info via subprocess (works in any environment, not just Colab)
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        for line in result.stdout.strip().split("\n"):
            print(f"GPU: {line}")
    else:
        print("No GPU detected (nvidia-smi not available or no CUDA device)")
except FileNotFoundError:
    print("nvidia-smi not found — running on CPU-only environment")

ram_gb = psutil.virtual_memory().total / 1e9
print(f"System RAM: {ram_gb:.1f} GB")

## 3. Pipeline 建立（2026 統一慣例）

2026 統一寫法使用 `device_map='auto'` 自動分配裝置：

- 有 GPU → 自動放 GPU
- 多 GPU → 自動分層（model parallelism）
- VRAM 不足 → 自動 offload 到 CPU/Disk

### 為何選 `torch_dtype=torch.bfloat16`？

| 精度 | 優點 | 缺點 |
|---|---|---|
| float32 (fp32) | 精度最高、支援最廣 | 記憶體需求 2x，推論速度慢 |
| float16 (fp16) | 記憶體省 50% | 數值範圍窄，梯度容易下溢（underflow）|
| bfloat16 (bf16) | 與 fp32 相同的指數範圍，數值穩定 | 需 Ampere (A100/RTX 30xx) 以上 GPU |

**結論**：bf16 在現代 GPU 上是最佳預設選擇，訓練和推論都適用。

In [ ]:
import torch
from transformers import pipeline

# 2026 unified convention: device_map='auto', no integer device
# safetensors=True: faster load, no pickle security risk
pipe = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Model device: {pipe.device}")
print(f"Model dtype : {next(pipe.model.parameters()).dtype}")

In [ ]:
# Single inference
result = pipe("this is good!")
print(result)

In [ ]:
# Batch inference: pass a list to get a list of results
batch = [
    "this is good!",
    "the movie was terrible and boring",
    "an average experience, nothing special",
]
results = pipe(batch)
for text, res in zip(batch, results):
    print(f"  [{res['label']:8s} {res['score']:.2f}]  {text}")

## 4. 指定中文模型

`pipeline` 的 `model` 參數接受任何 HuggingFace Hub 的 model id，可以無縫切換語言或領域專屬模型。

In [ ]:
pipe_zh = pipeline(
    "text-classification",
    model="uer/roberta-base-finetuned-chinanews-chinese",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

test_sentences = [
    "中國宣布開放兩岸邊境",
    "台積電發布最新季度財報，獲利創歷史新高",
    "奧運馬拉松項目中，肯亞選手奪得金牌",
]

for s in test_sentences:
    result = pipe_zh(s)
    print(f"[{result[0]['label']}] {s}")

## 5. 分段載入：model + tokenizer 分開下載

當你需要對 tokenizer 或 model 進行客製化設定時，可以分開載入再組合成 pipeline。

2026 統一寫法在 `from_pretrained` 時直接傳入 `device_map` 與 `torch_dtype`，讓框架在載入階段即決定精度與裝置，避免額外的記憶體複製。

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_id = "uer/roberta-base-finetuned-chinanews-chinese"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2026: specify dtype and device at load time
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

pipe_v3 = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
)

print(pipe_v3("中國宣布開放兩岸邊境"))

### safetensors 是什麼？

| 格式 | 載入方式 | 安全性 | 速度 |
|---|---|---|---|
| `.bin` (PyTorch pickle) | `torch.load` | 有 pickle 注入風險 | 較慢 |
| `.safetensors` | 直接 mmap | 無任意程式碼執行風險 | 快 2-5x |

HuggingFace Hub 上的新模型幾乎都提供 safetensors 格式。`use_safetensors=True` 會優先選用，Hub 上有 safetensors 時這已是預設行為（transformers 4.46+）。

## 6. CPU vs. GPU 效能比較

2026 寫法使用 `device_map` 參數切換裝置，傳入字串 `"cpu"` 或 `"auto"` 以指定目標硬體。

> **注意**：distilbert 這類小模型的 GPU 加速效益受限於 batch size 與 I/O 開銷。真正的加速倍數在大模型（>1B 參數）或批量推論時才顯著。

In [ ]:
import time
import torch

test_input = "this is good!"
n_runs = 50

def benchmark_pipeline(device_map_value: str, label: str) -> float:
    """Return mean inference time in milliseconds."""
    p = pipeline(
        "text-classification",
        model="distilbert-base-uncased-finetuned-sst-2-english",
        device_map=device_map_value,
        torch_dtype=torch.bfloat16,
    )
    times = []
    # Warm-up run
    p(test_input)
    for _ in range(n_runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        p(test_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    mean_ms = sum(times) / len(times)
    print(f"{label:6s}: {mean_ms:.2f} ms / call (n={n_runs})")
    return mean_ms

cpu_ms = benchmark_pipeline("cpu", "CPU")
if torch.cuda.is_available():
    gpu_ms = benchmark_pipeline("auto", "GPU")
    print(f"Speed-up ratio: {cpu_ms / gpu_ms:.1f}x")
else:
    print("GPU not available — skipping GPU benchmark")

## 7. Pipeline 參數詳解：以問答（QA）任務為例

In [ ]:
from transformers import QuestionAnsweringPipeline

# Load Chinese extractive QA pipeline
qa_pipe = pipeline(
    "question-answering",
    model="uer/roberta-base-chinese-extractive-qa",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Model device : {qa_pipe.device}")
print(f"Model dtype  : {next(qa_pipe.model.parameters()).dtype}")
print(f"Pipeline type: {type(qa_pipe).__name__}")

In [ ]:
# Inspect available parameters
help(QuestionAnsweringPipeline.__call__)

In [ ]:
question = "誰是威廉·莎士比亞的妻子？"
context = (
    "威廉·莎士比亞（William Shakespeare）是一位著名的文學家和劇作家，"
    "被認為是英國文學史上最偉大的作家之一。"
    "他的妻子名叫安妮·海瑟薇（Anne Hathaway），兩人於 1582 年結婚。"
)

answer = qa_pipe(
    question=question,
    context=context,
    max_answer_len=20,
)
print(f"Question : {question}")
print(f"Answer   : {answer['answer']}")
print(f"Score    : {answer['score']:.4f}")
print(f"Position : start={answer['start']}, end={answer['end']}")

## 8. 多模態 Pipeline：零樣本物體偵測（OWL-ViT）

### AutoProcessor：tokenizer 的多模態延伸

純文字任務只需要 `tokenizer`（文字 → `input_ids`）。
多模態任務（影像 + 文字）需要 `processor`，它同時處理：

- **文字側**：`tokenizer` → `input_ids`, `attention_mask`
- **影像側**：`image_processor` → `pixel_values`（調整大小、正規化、轉 tensor）

```
AutoProcessor
├── tokenizer      → input_ids, attention_mask
└── image_processor → pixel_values
```

這個抽象是通往第 05 模組（多模態）的關鍵橋樑：無論是 CLIP、BLIP、LLaVA、還是 Qwen-VL，都用同一套 `processor` 介面。

In [ ]:
import requests
from PIL import Image
from transformers import AutoProcessor, pipeline

model_id = "google/owlvit-base-patch32"

# Explicitly load the processor to show the multimodal abstraction
# processor = tokenizer + image_processor
processor = AutoProcessor.from_pretrained(model_id)
print(f"Processor type: {type(processor).__name__}")
print(f"  tokenizer       : {type(processor.tokenizer).__name__}")
print(f"  image_processor : {type(processor.image_processor).__name__}")

# Build the pipeline using the same model_id
# VRAM note: OWL-ViT base is ~330 MB — runs on any modern GPU
detector = pipeline(
    task="zero-shot-object-detection",
    model=model_id,
    device_map="auto",
)

In [ ]:
# Load a sample image
url = (
    "https://unsplash.com/photos/oj0zeY2Ltk4/download"
    "?ixid=MnwxMjA3fDB8MXxzZWFyY2h8MTR8fHBpY25pY3xlbnwwfHx8fDE2Nzc0OTE1NDk"
    "&force=true&w=640"
)
img = Image.open(requests.get(url, stream=True).raw).convert("RGB")
print(f"Image size: {img.size}")
img

In [ ]:
# Zero-shot detection: no fine-tuning needed, just supply candidate labels
predictions = detector(
    img,
    candidate_labels=["hat", "sunglasses", "book"],
)

for p in predictions:
    print(
        f"  {p['label']:12s} score={p['score']:.3f}  "
        f"box=[{p['box']['xmin']}, {p['box']['ymin']}, "
        f"{p['box']['xmax']}, {p['box']['ymax']}]"
    )

In [ ]:
from PIL import ImageDraw, ImageFont

img_draw = img.copy()
draw = ImageDraw.Draw(img_draw)

for p in predictions:
    box = p["box"]
    label = p["label"]
    score = p["score"]
    xmin, ymin, xmax, ymax = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
    draw.rectangle((xmin, ymin, xmax, ymax), outline="red", width=2)
    draw.text((xmin + 2, ymin + 2), f"{label}: {score:.2f}", fill="red")

img_draw

### 為什麼多模態需要 processor？

OWL-ViT 的輸入由兩部分組成：

1. **`input_ids`**（候選標籤的文字 token）：`"hat"` → `[101, 6045, 102]`
2. **`pixel_values`**（影像的 tensor）：`(1, 3, 768, 768)` 的正規化 float tensor

`AutoProcessor` 同時處理這兩條輸入流，輸出一個可直接送進模型的字典。若你只用 `tokenizer`，就只能處理文字部分，影像側會完全遺漏。

這個設計在 LLaVA、Qwen-VL、Gemma-VL 等 2026 主流視覺語言模型中同樣適用——它們全部使用 `AutoProcessor`。

## 9. Pipeline 底層解析：三步驟手動還原

理解底層有助於除錯與客製化。以下將 `pipeline` 的自動流程拆解成三個明確步驟。

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

model_id = "uer/roberta-base-finetuned-dianping-chinese"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Model device: {next(model.parameters()).device}")
print(f"Labels: {model.config.id2label}")

In [ ]:
# Step 1: Tokenize
# The tokenizer converts raw text into numeric token IDs
input_text = "這家素食餐廳很好吃"
inputs = tokenizer(input_text, return_tensors="pt")

# Move inputs to the same device as the model
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

print("=== Tokenizer output ===")
for k, v in inputs.items():
    print(f"  {k:20s}: shape={v.shape}, values={v[0].tolist()}")

### tokenizer 輸出欄位說明

| 欄位 | 形狀 | 說明 |
|---|---|---|
| `input_ids` | `(batch, seq_len)` | 每個 token 對應詞彙表中的整數 ID。`101`=[CLS]，`102`=[SEP] |
| `token_type_ids` | `(batch, seq_len)` | 區分句子 A（0）與句子 B（1），單句輸入全為 0 |
| `attention_mask` | `(batch, seq_len)` | 真實 token 標 1，padding token 標 0；self-attention 計算時只關注標 1 的位置 |

In [ ]:
# Step 2: Model forward pass
# Output logits are raw (unnormalized) scores before softmax
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(f"Logits shape: {logits.shape}")
print(f"Logits (raw): {logits}")

In [ ]:
# Step 3: Post-process — convert logits to probabilities
probs = torch.softmax(logits.float(), dim=-1)  # cast to fp32 for numerical stability
pred_idx = torch.argmax(probs, dim=-1).item()
pred_label = model.config.id2label[pred_idx]
pred_score = probs[0, pred_idx].item()

print(f"Input   : {input_text}")
print(f"Label   : {pred_label}")
print(f"Score   : {pred_score:.4f}")
print()
for idx, prob in enumerate(probs[0].tolist()):
    label = model.config.id2label[idx]
    bar = '#' * int(prob * 40)
    print(f"  {label:12s} {prob:.4f} {bar}")

In [ ]:
# Verify: pipeline should produce the same result
build_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
)
pipe_result = build_pipe(input_text)
print(f"Pipeline result: {pipe_result}")
print(f"Manual  result: label={pred_label}, score={pred_score:.4f}")

In [ ]:
# model.config stores all metadata: labels, architecture, vocab size, etc.
# id2label / label2id are persisted when you call save_pretrained()
print("=== model.config (selected fields) ===")
print(f"  architecture   : {model.config.architectures}")
print(f"  hidden_size    : {model.config.hidden_size}")
print(f"  num_labels     : {model.config.num_labels}")
print(f"  id2label       : {model.config.id2label}")
print(f"  label2id       : {model.config.label2id}")

### 三步驟與 pipeline 的對應關係

```
原始輸入 (str / PIL.Image)
     |
     v  tokenizer / image_processor / processor
 token tensors: input_ids, attention_mask, pixel_values ...
     |
     v  model.forward(**inputs)
 outputs.logits / outputs.last_hidden_state
     |
     v  post-processing (softmax, argmax, NMS, ...)
 結構化結果 dict: {'label': ..., 'score': ...}
```

`pipeline` 自動完成這三步，但理解底層讓你能夠：
- 客製化 tokenizer 參數（padding 策略、max_length）
- 在 forward pass 之間插入自訂邏輯
- 用 `model.config.id2label` 還原真實標籤

## 10. 小結與練習

### 本節重點回顧

1. `pipeline` 封裝了 tokenize → forward → post-process 三步驟，是最快的推論起點
2. 2026 統一裝置慣例：`device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True`
3. `AutoProcessor = tokenizer + image_processor`，是多模態任務的統一入口
4. `model.config.id2label` 是模型輸出索引到真實標籤的映射，應隨模型一起持久化

### 練習題

1. **基礎**：用 `pipeline('fill-mask', model='bert-base-chinese')` 預測「台灣是[MASK]的寶島」中遮蔽詞的前五個候選詞
2. **進階**：將第 9 節的三步驟手動流程改為批次輸入（`input_text = ['好吃', '難吃', '普通']`），並觀察 `tokenizer` 如何自動 padding
3. **延伸**：閱讀 `ZeroShotObjectDetectionPipeline` 原始碼（`transformers/pipelines/zero_shot_object_detection.py`），找出 `preprocess` 方法中如何呼叫 `processor`，理解 processor 的雙流輸入

### 下一步

- `../02tokenizer/01.tokenizer.ipynb`：深入 tokenizer 的 BPE/WordPiece 原理
- `../03model/`：手動載入各種 Auto 模型類別
- `../../05-Multimodal/`：AutoProcessor 的完整多模態應用